In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip install keras-preprocessing

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.8 MB/s eta 0:00:00


In [ ]:
%pip install pymupdf
%pip install spacy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 48.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 42.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import fitz  # PyMuPDF for PDF text extraction
import spacy
import torch
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification, AutoTokenizer, AutoModelForSequenceClassification
from tensorflow.keras.preprocessing.text import text_to_word_sequence
from sklearn.feature_extraction.text import CountVectorizer
import pickle
import pandas as pd
import os
import json
import re
import torch.nn.functional as F

In [ ]:
# @title
# Load the pre-trained BERT model for climate-related classification
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")  # Or use a climate-specific BERT model
model = TFBertForSequenceClassification.from_pretrained("bert-base-uncased")  # Replace with climate-specific model

# Load bigrams for different categories (Climate, Regulatory, Opportunity, and Physical)
cc_bigrams = pickle.load(open('/content/drive/MyDrive/BTP/Replication/B/bigrams/bigrams_07222021.pkl', 'rb'))
cc_bigrams = [x.lower() for x in cc_bigrams]

rg_bgs = pickle.load(open('/content/drive/MyDrive/BTP/Replication/B/bigrams/regulatory_bigrams_4.pkl', 'rb'))
op_bgs = pickle.load(open('/content/drive/MyDrive/BTP/Replication/B/bigrams/opportunity_bigrams_4.pkl', 'rb'))
ph_bgs = pickle.load(open('/content/drive/MyDrive/BTP/Replication/B/bigrams/physical_bigrams_4.pkl', 'rb'))

# Initialize the CountVectorizer for bigrams
vectorizer = CountVectorizer(analyzer='word', ngram_range=(2, 2), stop_words='english')

nlp = spacy.load('en_core_web_sm')

In [ ]:
# @title
# Step 1: Extract text from PDF
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text")
    return text

# Step 2: Preprocess the text (cleaning and sentence segmentation)
def preprocess_text(text):
    # Tokenize and preprocess using Keras text_to_word_sequence
    words = text_to_word_sequence(text, lower=True, split=" ", filters="")
    return " ".join(words)

# Step 3: Calculate total number of bigrams
def calculate_total_bigrams(text):
    # Tokenize the text into sentences using spaCy
    doc = nlp(text)
    sentences = [sent.text for sent in doc.sents]

    # Calculate bigrams using CountVectorizer
    bigrams = vectorizer.fit_transform(sentences)
    total_bigrams = bigrams.sum()  # Sum of all bigrams in the document
    return total_bigrams, sentences

# Step 4: Label sentences as climate-related or not using BERT
def classify_climate_sentence(sentence):
    inputs = tokenizer(sentence, return_tensors="tf", truncation=True, padding=True, max_length=512)
    outputs = model(**inputs)
    logits = outputs.logits
    prediction = np.argmax(logits.numpy(), axis=-1)
    return prediction[0] == 1  # Assuming '1' means climate-related, '0' means not

# Step 5: Label all sentences and filter climate-related ones
def label_climate_sentences(sentences):
    climate_sentences = []
    for sentence in sentences:
        if classify_climate_sentence(sentence):
            climate_sentences.append(sentence)
    return climate_sentences

# Step 6: Extract bigrams from climate-related sentences
def extract_climate_related_bigrams(climate_sentences):
    # Calculate bigrams for climate-related sentences
    climate_bigrams = vectorizer.transform(climate_sentences)
    climate_bigrams_count = climate_bigrams.sum()  # Sum of climate-related bigrams

    # Convert to list of bigrams for further frequency calculation
    climate_bigrams_list = vectorizer.get_feature_names_out()
    return climate_bigrams_count, climate_bigrams_list

# Step 7: Calculate the frequency of bigrams from pickle files (CC, RG, OP, PH)
def calculate_bigram_frequencies(bigrams_list, bigram_set):
    # Check frequency of bigrams in the list from pickle file
    bigram_frequency = {}
    for bigram in bigrams_list:
        if bigram in bigram_set:
            bigram_frequency[bigram] = bigram_frequency.get(bigram, 0) + 1
    return bigram_frequency

# Step 8: Calculate exposure
def calculate_exposure(bigrams_frequency, total_bigrams):
    # Exposure is calculated as the sum of frequencies of matching bigrams divided by total bigrams
    exposure = sum(bigrams_frequency.values()) / total_bigrams if total_bigrams > 0 else 0
    return exposure

# Main function to run the entire process
def analyze_pdf(pdf_path, transcript_id):
    # Step 1: Extract text from PDF
    text = extract_text_from_pdf(pdf_path)

    # Step 2: Preprocess text
    preprocessed_text = preprocess_text(text)

    # Step 3: Calculate total number of bigrams and get sentences
    total_bigrams, sentences = calculate_total_bigrams(preprocessed_text)

    # Step 4: Label climate-related sentences
    climate_sentences = label_climate_sentences(sentences)

    # Step 5: Extract bigrams from climate-related sentences
    climate_bigrams_count, climate_bigrams_list = extract_climate_related_bigrams(climate_sentences)

    # Step 6: Calculate bigram frequencies for climate-related bigrams, regulatory, opportunity, and physical bigrams
    cc_bigrams_frequency = calculate_bigram_frequencies(climate_bigrams_list, cc_bigrams)
    rg_bigrams_frequency = calculate_bigram_frequencies(climate_bigrams_list, rg_bgs)
    op_bigrams_frequency = calculate_bigram_frequencies(climate_bigrams_list, op_bgs)
    ph_bigrams_frequency = calculate_bigram_frequencies(climate_bigrams_list, ph_bgs)

    # Step 7: Calculate exposure ratio for climate-related bigrams
    cc_exposure = calculate_exposure(cc_bigrams_frequency, total_bigrams)
    op_exposure = calculate_exposure(op_bigrams_frequency, total_bigrams)
    rg_exposure = calculate_exposure(rg_bigrams_frequency, total_bigrams)
    ph_exposure = calculate_exposure(ph_bigrams_frequency, total_bigrams)

    # Store results in the dataframe in the required format with dictionaries of bigrams and their frequencies
    new_row = {
        'Transcript ID': transcript_id,
        'CCExposure': cc_exposure,
        'Climate Change Bigrams': cc_bigrams_frequency,  # Store as dictionary: bigrams and their frequencies
        'Opportunity Bigrams': op_bigrams_frequency,  # Store as dictionary: bigrams and their frequencies
        'Regulatory Bigrams': rg_bigrams_frequency,  # Store as dictionary: bigrams and their frequencies
        'Physical Bigrams': ph_bigrams_frequency,  # Store as dictionary: bigrams and their frequencies
        'OpExposure': op_exposure,
        'RegExposure': rg_exposure,
        'PhExposure': ph_exposure
    }

    return new_row

# Initialize a DataFrame to store all the results
consolidated_df = pd.DataFrame(columns=[
    'Transcript ID', 'CCExposure', 'Climate Change Bigrams',
    'Opportunity Bigrams', 'Regulatory Bigrams', 'Physical Bigrams',
    'OpExposure', 'RegExposure', 'PhExposure'
])

# Load the transcripts from directory (assumed to be PDF files)
tppath = '/content/drive/MyDrive/sbi/Transcripts'
tp_list = [file for file in os.listdir(tppath) if file.endswith('.pdf')]  # Look for PDF files
tp_list.sort()

# Process each PDF file
for tp_file in tp_list:
    tp_path = os.path.join(tppath, tp_file)

    # Extract the transcript ID from the filename (or assume the file name is the transcript ID)
    transcript_id = os.path.splitext(tp_file)[0]

    # Analyze the PDF file and generate the new row
    new_row = analyze_pdf(tp_path, transcript_id)

    # Append the new row to the DataFrame
    consolidated_df = pd.concat([consolidated_df, pd.DataFrame([new_row])], ignore_index=True)

    print(transcript_id)

# Step 9: Save the consolidated data to a CSV file
file_name = '/content/drive/MyDrive/BTP/sbi_transcripts.csv'
consolidated_df.to_csv(file_name, index=False)

print(f"Data successfully saved to {file_name}.")

In [ ]:
# Load the pre-trained ClimateBERT model for climate-related classification
tokenizer_climate = AutoTokenizer.from_pretrained("climatebert/distilroberta-base-climate-sentiment")
model_climate = AutoModelForSequenceClassification.from_pretrained("climatebert/distilroberta-base-climate-sentiment")

# Load bigrams for different categories (Climate, Regulatory, Opportunity, and Physical)
cc_bigrams_climate = pickle.load(open('/content/drive/MyDrive/BTP/Replication/B/bigrams/bigrams_07222021.pkl', 'rb'))
cc_bigrams_climate = [x.lower() for x in cc_bigrams_climate]

rg_bgs_climate = pickle.load(open('/content/drive/MyDrive/BTP/Replication/B/bigrams/regulatory_bigrams_4.pkl', 'rb'))
op_bgs_climate = pickle.load(open('/content/drive/MyDrive/BTP/Replication/B/bigrams/opportunity_bigrams_4.pkl', 'rb'))
ph_bgs_climate = pickle.load(open('/content/drive/MyDrive/BTP/Replication/B/bigrams/physical_bigrams_4.pkl', 'rb'))

# Initialize the CountVectorizer for bigrams
vectorizer_climate = CountVectorizer(analyzer='word', ngram_range=(2, 2), stop_words='english')

# Load spaCy model
nlp_climate = spacy.load('en_core_web_sm')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/4.48k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/947 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

In [ ]:
# @title
# Step 1: Extract text from PDF
def extract_text_from_pdf_climate(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text")
    return text

# Step 2: Preprocess the text (cleaning and sentence segmentation)
def preprocess_text_climate(text):
    # Tokenize and preprocess using Keras text_to_word_sequence
    words = text_to_word_sequence(text, lower=True, split=" ", filters="")
    return " ".join(words)

# Step 3: Calculate total number of bigrams
def calculate_total_bigrams_climate(text):
    # Tokenize the text into sentences using spaCy
    doc = nlp_climate(text)
    sentences = [sent.text for sent in doc.sents]

    # Calculate bigrams using CountVectorizer
    bigrams = vectorizer_climate.fit_transform(sentences)
    total_bigrams = bigrams.sum()  # Sum of all bigrams in the document
    return total_bigrams, sentences

# Step 4: Label sentences as climate-related or not using ClimateBERT
def classify_climate_sentence_climate(sentence):
    inputs = tokenizer_climate(sentence, return_tensors="pt", truncation=True, padding=True, max_length=512)  # Change 'tf' to 'pt' for PyTorch
    outputs = model_climate(**inputs)
    logits = outputs.logits
    prediction = np.argmax(logits.detach().numpy(), axis=-1)  # Detach and convert to NumPy for compatibility
    return prediction[0] == 1  # Assuming '1' means climate-related, '0' means not

# Step 5: Label all sentences and filter climate-related ones
def label_climate_sentences_climate(sentences):
    climate_sentences = []
    for sentence in sentences:
        if classify_climate_sentence_climate(sentence):
            climate_sentences.append(sentence)
    return climate_sentences

# Step 6: Extract bigrams from climate-related sentences
def extract_climate_related_bigrams_climate(climate_sentences):
    # Calculate bigrams for climate-related sentences
    climate_bigrams = vectorizer_climate.transform(climate_sentences)
    climate_bigrams_count = climate_bigrams.sum()  # Sum of climate-related bigrams

    # Convert to list of bigrams for further frequency calculation
    climate_bigrams_list = vectorizer_climate.get_feature_names_out()
    return climate_bigrams_count, climate_bigrams_list

# Step 7: Calculate the frequency of bigrams from pickle files (CC, RG, OP, PH)
def calculate_bigram_frequencies_climate(bigrams_list, bigram_set):
    # Check frequency of bigrams in the list from pickle file
    bigram_frequency = {}
    for bigram in bigrams_list:
        if bigram in bigram_set:
            bigram_frequency[bigram] = bigram_frequency.get(bigram, 0) + 1
    return bigram_frequency

# Step 8: Calculate exposure
def calculate_exposure_climate(bigrams_frequency, total_bigrams):
    # Exposure is calculated as the sum of frequencies of matching bigrams divided by total bigrams
    exposure = sum(bigrams_frequency.values()) / total_bigrams if total_bigrams > 0 else 0
    return exposure

# Main function to run the entire process
def analyze_pdf_climate(pdf_path, transcript_id):
    # Step 1: Extract text from PDF
    text = extract_text_from_pdf_climate(pdf_path)

    # Step 2: Preprocess text
    preprocessed_text = preprocess_text_climate(text)

    # Step 3: Calculate total number of bigrams and get sentences
    total_bigrams, sentences = calculate_total_bigrams_climate(preprocessed_text)

    # Step 4: Label climate-related sentences
    climate_sentences = label_climate_sentences_climate(sentences)

    # Step 5: Extract bigrams from climate-related sentences
    climate_bigrams_count, climate_bigrams_list = extract_climate_related_bigrams_climate(climate_sentences)

    # Step 6: Calculate bigram frequencies for climate-related bigrams, regulatory, opportunity, and physical bigrams
    cc_bigrams_frequency = calculate_bigram_frequencies_climate(climate_bigrams_list, cc_bigrams_climate)
    rg_bigrams_frequency = calculate_bigram_frequencies_climate(climate_bigrams_list, rg_bgs_climate)
    op_bigrams_frequency = calculate_bigram_frequencies_climate(climate_bigrams_list, op_bgs_climate)
    ph_bigrams_frequency = calculate_bigram_frequencies_climate(climate_bigrams_list, ph_bgs_climate)

    # Step 7: Calculate exposure ratio for climate-related bigrams
    cc_exposure = calculate_exposure_climate(cc_bigrams_frequency, total_bigrams)
    op_exposure = calculate_exposure_climate(op_bigrams_frequency, total_bigrams)
    rg_exposure = calculate_exposure_climate(rg_bigrams_frequency, total_bigrams)
    ph_exposure = calculate_exposure_climate(ph_bigrams_frequency, total_bigrams)

    # Store results in the dataframe in the required format with dictionaries of bigrams and their frequencies
    new_row = {
        'Transcript ID': transcript_id,
        'CCExposure': cc_exposure,
        'Climate Change Bigrams': cc_bigrams_frequency,  # Store as dictionary: bigrams and their frequencies
        'Opportunity Bigrams': op_bigrams_frequency,  # Store as dictionary: bigrams and their frequencies
        'Regulatory Bigrams': rg_bigrams_frequency,  # Store as dictionary: bigrams and their frequencies
        'Physical Bigrams': ph_bigrams_frequency,  # Store as dictionary: bigrams and their frequencies
        'OpExposure': op_exposure,
        'RegExposure': rg_exposure,
        'PhExposure': ph_exposure
    }

    return new_row

# Initialize a DataFrame to store all the results
consolidated_df = pd.DataFrame(columns=[
    'Transcript ID', 'CCExposure', 'Climate Change Bigrams',
    'Opportunity Bigrams', 'Regulatory Bigrams', 'Physical Bigrams',
    'OpExposure', 'RegExposure', 'PhExposure'
])

# Load the transcripts from directory (assumed to be PDF files)
tppath = '/content/drive/MyDrive/sbi/AR'
tp_list = [file for file in os.listdir(tppath) if file.endswith('.pdf')]  # Look for PDF files
tp_list.sort()

# Process each PDF file
for tp_file in tp_list:
    tp_path = os.path.join(tppath, tp_file)

    # Extract the transcript ID from the filename (or assume the file name is the transcript ID)
    transcript_id = os.path.splitext(tp_file)[0]

    # Analyze the PDF file and generate the new row
    new_row = analyze_pdf_climate(tp_path, transcript_id)

    # Append the new row to the DataFrame
    consolidated_df = pd.concat([consolidated_df, pd.DataFrame([new_row])], ignore_index=True)

    print(transcript_id)


# Step 9: Save the consolidated data to a CSV file
file_name = '/content/drive/MyDrive/BTP/sbi_ar_climate.csv'
consolidated_df.to_csv(file_name, index=False)

print(f"Data successfully saved to {file_name}.")

In [ ]:
# Function to extract text from a PDF
def extract_text_from_pdf_climate(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text")
    return text

# Function to preprocess text
def preprocess_text_climate(text):
    text = text.replace("\n", " ")  # Remove newline characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove non-alphanumeric characters
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    return text

# Function to classify a sentence using ClimateBERT
def classify_climate_sentence_climate(sentence):
    inputs = tokenizer_climate(sentence, return_tensors="pt", truncation=True, padding=True, max_length=512)
    outputs = model_climate(**inputs)
    logits = outputs.logits
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    return probabilities[0][1].item() > 0.5  # Return True if climate-related

# Function to extract bigrams and calculate their frequencies
def extract_and_calculate_bigram_frequencies(preprocessed_sentences, bigram_set):
    vectorizer_climate.fit(preprocessed_sentences)  # Fit vectorizer on sentences
    bigrams_matrix = vectorizer_climate.transform(preprocessed_sentences)
    bigrams_list = vectorizer_climate.get_feature_names_out()

    # Calculate frequencies for bigrams in the given set
    bigram_frequency = {}
    for idx, bigram in enumerate(bigrams_list):
        if bigram in bigram_set:
            bigram_frequency[bigram] = int(bigrams_matrix[:, idx].sum())  # Sum occurrences
    return bigram_frequency

# Function to aggregate bigrams across reports
def aggregate_bigrams_frequencies(aggregated_bigrams, bigrams_frequency):
    for bigram, freq in bigrams_frequency.items():
        if bigram in aggregated_bigrams:
            aggregated_bigrams[bigram] += freq
        else:
            aggregated_bigrams[bigram] = freq

# Function to calculate exposure ratios
def calculate_exposure_ratios(cc_bigrams_frequency, op_bigrams_frequency, rg_bigrams_frequency, ph_bigrams_frequency, total_bigrams):
    cc_exposure = sum(cc_bigrams_frequency.values()) / total_bigrams if total_bigrams > 0 else 0
    op_exposure = sum(op_bigrams_frequency.values()) / total_bigrams if total_bigrams > 0 else 0
    rg_exposure = sum(rg_bigrams_frequency.values()) / total_bigrams if total_bigrams > 0 else 0
    ph_exposure = sum(ph_bigrams_frequency.values()) / total_bigrams if total_bigrams > 0 else 0
    return cc_exposure, op_exposure, rg_exposure, ph_exposure

# Function to analyze a single PDF
def analyze_pdf_climate(pdf_path, transcript_id, all_reports_bigrams):
    text = extract_text_from_pdf_climate(pdf_path)  # Extract text
    sentences = text.split('.')  # Split into sentences
    sentences = [preprocess_text_climate(sentence) for sentence in sentences]  # Preprocess sentences

    # Classify climate-related sentences
    climate_sentences = [s for s in sentences if classify_climate_sentence_climate(s)]

    # Fit vectorizer on full text and calculate total bigrams
    vectorizer_climate.fit(sentences)
    total_bigrams = sum(vectorizer_climate.transform(sentences).sum(axis=0).tolist()[0])

    # Calculate bigram frequencies for each category
    cc_bigrams_frequency = extract_and_calculate_bigram_frequencies(climate_sentences, cc_bigrams_climate)
    op_bigrams_frequency = extract_and_calculate_bigram_frequencies(climate_sentences, op_bgs_climate)
    rg_bigrams_frequency = extract_and_calculate_bigram_frequencies(climate_sentences, rg_bgs_climate)
    ph_bigrams_frequency = extract_and_calculate_bigram_frequencies(climate_sentences, ph_bgs_climate)

    # Aggregate frequencies across reports
    aggregate_bigrams_frequencies(all_reports_bigrams['cc_bigrams'], cc_bigrams_frequency)
    aggregate_bigrams_frequencies(all_reports_bigrams['op_bigrams'], op_bigrams_frequency)
    aggregate_bigrams_frequencies(all_reports_bigrams['rg_bigrams'], rg_bigrams_frequency)
    aggregate_bigrams_frequencies(all_reports_bigrams['ph_bigrams'], ph_bigrams_frequency)

    # Calculate exposure ratios
    cc_exposure, op_exposure, rg_exposure, ph_exposure = calculate_exposure_ratios(
        cc_bigrams_frequency, op_bigrams_frequency, rg_bigrams_frequency, ph_bigrams_frequency, total_bigrams
    )
    print(cc_exposure)

    # Create a row for the transcript
    return {
        'Transcript ID': transcript_id,
        'CCExposure': cc_exposure,
        'Opportunity Exposure': op_exposure,
        'Regulatory Exposure': rg_exposure,
        'Physical Exposure': ph_exposure,
        'Climate Change Bigrams': cc_bigrams_frequency,
        'Opportunity Bigrams': op_bigrams_frequency,
        'Regulatory Bigrams': rg_bigrams_frequency,
        'Physical Bigrams': ph_bigrams_frequency
    }

# Initialize DataFrame and aggregated bigrams
consolidated_df = pd.DataFrame(columns=[
    'Transcript ID', 'CCExposure', 'Opportunity Exposure',
    'Regulatory Exposure', 'Physical Exposure',
    'Climate Change Bigrams', 'Opportunity Bigrams',
    'Regulatory Bigrams', 'Physical Bigrams'
])
all_reports_bigrams = {'cc_bigrams': {}, 'op_bigrams': {}, 'rg_bigrams': {}, 'ph_bigrams': {}}

# Directory of PDFs
pdf_folder = '/content/drive/MyDrive/sbi/SR'

# Analyze all PDFs
for filename in os.listdir(pdf_folder):
    if filename.endswith('.pdf'):
        pdf_path = os.path.join(pdf_folder, filename)
        transcript_id = filename.split('.')[0]
        new_row = analyze_pdf_climate(pdf_path, transcript_id, all_reports_bigrams)
        consolidated_df = pd.concat([consolidated_df, pd.DataFrame([new_row])], ignore_index=True)

# Save consolidated results
consolidated_df.to_csv('/content/drive/MyDrive/BTP/sr_climate_analysis_sentences_results.csv', index=False)

# Save top 100 bigrams for each category
def save_top_100_bigrams(all_reports_bigrams, category, filename):
    bigrams = all_reports_bigrams[category]
    sorted_bigrams = sorted(bigrams.items(), key=lambda x: x[1], reverse=True)
    top_100_bigrams = sorted_bigrams[:100]
    pd.DataFrame(top_100_bigrams, columns=['Bigram', 'Frequency']).to_csv(filename, index=False)

save_top_100_bigrams(all_reports_bigrams, 'cc_bigrams', '/content/drive/MyDrive/BTP/cc_bigrams_sentences_top_100_sent.csv')
save_top_100_bigrams(all_reports_bigrams, 'op_bigrams', '/content/drive/MyDrive/BTP/op_bigrams_sentences_top_100_sent.csv')
save_top_100_bigrams(all_reports_bigrams, 'rg_bigrams', '/content/drive/MyDrive/BTP/rg_bigrams_sentences_top_100_sent.csv')
save_top_100_bigrams(all_reports_bigrams, 'ph_bigrams', '/content/drive/MyDrive/BTP/ph_bigrams_sentences_top_100_sent.csv')

0.0015207084712407191


<ipython-input-6-e17e9922668d>:116: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  consolidated_df = pd.concat([consolidated_df, pd.DataFrame([new_row])], ignore_index=True)


0.0016783643576441867
0.0038327773440272554
0.003021664766248575
0.0026471700491617293
0.0017736786094359701
0.0025789068514241727
0.0026626612958284973


In [ ]:
# Function to extract text from a PDF and split into paragraphs
def extract_text_from_pdf_climate(pdf_path):
    doc = fitz.open(pdf_path)
    paragraphs = []
    for page in doc:
        text = page.get_text("blocks")  # Extract blocks of text
        page_paragraphs = [block[4] for block in text if block[4].strip()]  # Extract non-empty blocks
        paragraphs.extend(page_paragraphs)
    return paragraphs

# Function to preprocess text
def preprocess_text_climate(text):
    text = text.replace("\n", " ")  # Remove newline characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove non-alphanumeric characters
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    return text

# Function to classify a paragraph using ClimateBERT
def classify_climate_paragraph_climate(paragraph):
    inputs = tokenizer_climate(paragraph, return_tensors="pt", truncation=True, padding=True, max_length=512)
    outputs = model_climate(**inputs)
    logits = outputs.logits
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    return probabilities[0][1].item() > 0.5  # Return True if climate-related

# Function to extract bigrams and calculate their frequencies
def extract_and_calculate_bigram_frequencies(preprocessed_paragraphs, bigram_set):
    vectorizer_climate.fit(preprocessed_paragraphs)  # Fit vectorizer on paragraphs
    bigrams_matrix = vectorizer_climate.transform(preprocessed_paragraphs)
    bigrams_list = vectorizer_climate.get_feature_names_out()

    # Calculate frequencies for bigrams in the given set
    bigram_frequency = {}
    for idx, bigram in enumerate(bigrams_list):
        if bigram in bigram_set:
            bigram_frequency[bigram] = int(bigrams_matrix[:, idx].sum())  # Sum occurrences
    return bigram_frequency

# Function to aggregate bigrams across reports
def aggregate_bigrams_frequencies(aggregated_bigrams, bigrams_frequency):
    for bigram, freq in bigrams_frequency.items():
        if bigram in aggregated_bigrams:
            aggregated_bigrams[bigram] += freq
        else:
            aggregated_bigrams[bigram] = freq

# Function to calculate exposure ratios
def calculate_exposure_ratios(cc_bigrams_frequency, op_bigrams_frequency, rg_bigrams_frequency, ph_bigrams_frequency, total_bigrams):
    cc_exposure = sum(cc_bigrams_frequency.values()) / total_bigrams if total_bigrams > 0 else 0
    op_exposure = sum(op_bigrams_frequency.values()) / total_bigrams if total_bigrams > 0 else 0
    rg_exposure = sum(rg_bigrams_frequency.values()) / total_bigrams if total_bigrams > 0 else 0
    ph_exposure = sum(ph_bigrams_frequency.values()) / total_bigrams if total_bigrams > 0 else 0
    return cc_exposure, op_exposure, rg_exposure, ph_exposure

# Function to analyze a single PDF
def analyze_pdf_climate(pdf_path, transcript_id, all_reports_bigrams):
    paragraphs = extract_text_from_pdf_climate(pdf_path)  # Extract paragraphs
    paragraphs = [preprocess_text_climate(p) for p in paragraphs]  # Preprocess paragraphs

    # Classify climate-related paragraphs
    climate_paragraphs = [p for p in paragraphs if classify_climate_paragraph_climate(p)]

    # Fit vectorizer on full text and calculate total bigrams
    vectorizer_climate.fit(paragraphs)
    total_bigrams = sum(vectorizer_climate.transform(paragraphs).sum(axis=0).tolist()[0])

    # Calculate bigram frequencies for each category
    cc_bigrams_frequency = extract_and_calculate_bigram_frequencies(climate_paragraphs, cc_bigrams_climate)
    op_bigrams_frequency = extract_and_calculate_bigram_frequencies(climate_paragraphs, op_bgs_climate)
    rg_bigrams_frequency = extract_and_calculate_bigram_frequencies(climate_paragraphs, rg_bgs_climate)
    ph_bigrams_frequency = extract_and_calculate_bigram_frequencies(climate_paragraphs, ph_bgs_climate)

    # Aggregate frequencies across reports
    aggregate_bigrams_frequencies(all_reports_bigrams['cc_bigrams'], cc_bigrams_frequency)
    aggregate_bigrams_frequencies(all_reports_bigrams['op_bigrams'], op_bigrams_frequency)
    aggregate_bigrams_frequencies(all_reports_bigrams['rg_bigrams'], rg_bigrams_frequency)
    aggregate_bigrams_frequencies(all_reports_bigrams['ph_bigrams'], ph_bigrams_frequency)

    # Calculate exposure ratios
    cc_exposure, op_exposure, rg_exposure, ph_exposure = calculate_exposure_ratios(
        cc_bigrams_frequency, op_bigrams_frequency, rg_bigrams_frequency, ph_bigrams_frequency, total_bigrams
    )
    print(cc_exposure)
    # Create a row for the transcript
    return {
        'Transcript ID': transcript_id,
        'CCExposure': cc_exposure,
        'Opportunity Exposure': op_exposure,
        'Regulatory Exposure': rg_exposure,
        'Physical Exposure': ph_exposure,
        'Climate Change Bigrams': cc_bigrams_frequency,
        'Opportunity Bigrams': op_bigrams_frequency,
        'Regulatory Bigrams': rg_bigrams_frequency,
        'Physical Bigrams': ph_bigrams_frequency
    }

# Initialize DataFrame and aggregated bigrams
consolidated_df = pd.DataFrame(columns=[
    'Transcript ID', 'CCExposure', 'Opportunity Exposure',
    'Regulatory Exposure', 'Physical Exposure',
    'Climate Change Bigrams', 'Opportunity Bigrams',
    'Regulatory Bigrams', 'Physical Bigrams'
])
all_reports_bigrams = {'cc_bigrams': {}, 'op_bigrams': {}, 'rg_bigrams': {}, 'ph_bigrams': {}}

# Directory of PDFs
pdf_folder = '/content/drive/MyDrive/sbi/SR'

# Analyze all PDFs
for filename in os.listdir(pdf_folder):
    if filename.endswith('.pdf'):
        pdf_path = os.path.join(pdf_folder, filename)
        transcript_id = filename.split('.')[0]
        new_row = analyze_pdf_climate(pdf_path, transcript_id, all_reports_bigrams)
        consolidated_df = pd.concat([consolidated_df, pd.DataFrame([new_row])], ignore_index=True)

# Save consolidated results
consolidated_df.to_csv('/content/drive/MyDrive/BTP/sr_climate_analysis_para_results.csv', index=False)

# Save top 100 bigrams for each category
def save_top_100_bigrams(all_reports_bigrams, category, filename):
    bigrams = all_reports_bigrams[category]
    sorted_bigrams = sorted(bigrams.items(), key=lambda x: x[1], reverse=True)
    top_100_bigrams = sorted_bigrams[:100]
    pd.DataFrame(top_100_bigrams, columns=['Bigram', 'Frequency']).to_csv(filename, index=False)

save_top_100_bigrams(all_reports_bigrams, 'cc_bigrams', '/content/drive/MyDrive/BTP/cc_bigrams_sentences_top_100_para.csv')
save_top_100_bigrams(all_reports_bigrams, 'op_bigrams', '/content/drive/MyDrive/BTP/op_bigrams_sentences_top_100_para.csv')
save_top_100_bigrams(all_reports_bigrams, 'rg_bigrams', '/content/drive/MyDrive/BTP/rg_bigrams_sentences_top_100_para.csv')
save_top_100_bigrams(all_reports_bigrams, 'ph_bigrams', '/content/drive/MyDrive/BTP/ph_bigrams_sentences_top_100_para.csv')

0.0015621745469693814


<ipython-input-7-42fa36bd091d>:116: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  consolidated_df = pd.concat([consolidated_df, pd.DataFrame([new_row])], ignore_index=True)


0.0012894906511927789
0.0017395250340341854
0.0017594275995542784
0.0018979057591623036
0.001728181705962227
0.002432991362880662
0.0023389274691358024
